# 77館材化所冰水主機分析

確認兩台主機的運轉狀況與性能曲線，再分析負載、耗電與外氣條件的關係。

## 1. 環境設定

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# 切到專案根目錄，才 import 得到 data、model、research
if Path.cwd().name == "analysis":
    os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from data import weather
from data.sensor import is_running  # 運轉：功率 > 20 kW 且冰水流量 > 100 LPM
from data.sensor import load_chiller  # 抓一台主機的 5 個監測項目
from model import rules  # 性能曲線（同 onoff_control.ipynb）
from research import chiller

pio.renderers.default = "jupyterlab"

### 圖表統一格式

In [2]:
# 顏色、中文名稱、單位、版面統一放在 research/chart_style.py
from research.chart_style import (
    COLORS,
    GROUP_SHADES,
    LIGHT_BLUE,
    NAMES,
    REF_COLOR,
    UNITS,
    date_axis,
    label,
    show,
)

## 2. CH-1 / CH-2 運轉狀況

In [3]:
df_ch1 = await load_chiller("77館材化所CH1")
df_ch2 = await load_chiller("77館材化所CH2")
for chiller_name, chiller_data in [("CH-1", df_ch1), ("CH-2", df_ch2)]:
    print(
        f"{chiller_name}：",
        len(chiller_data),
        "筆，",
        chiller_data.index.min(),
        "~",
        chiller_data.index.max(),
    )

CH-1： 37294 筆， 2026-09-17 13:23:00+08:00 ~ 2026-10-13 11:24:00+08:00
CH-2： 37294 筆， 2026-09-17 13:23:00+08:00 ~ 2026-10-13 11:24:00+08:00


In [4]:
# 一台主機的水溫、功率、流量、冷凍噸時間序列
def plot_operation(name: str, df: pd.DataFrame) -> None:
    df = df.resample("1min").mean()  # 補齊缺漏的分鐘，資料中斷處線才會斷開

    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=("冰水溫度 (°C)", label("kW"), label("FLOW"), label("Q_COOL")),
    )
    # 水溫有兩條線、需要圖例，放第一排緊鄰右上角圖例
    column_rows = [("T_CHWR", 1), ("T_CHWS", 1), ("kW", 2), ("FLOW", 3), ("Q_COOL", 4)]
    for column, row in column_rows:
        fig.add_trace(
            go.Scattergl(
                x=df.index,
                y=df[column],
                mode="lines",
                name=NAMES[column],
                line=dict(width=1.5, color=COLORS[column]),
                showlegend=(row == 1),
            ),
            row=row,
            col=1,
        )

    fig.update_layout(hovermode="x unified")
    date_axis(fig)
    show(
        fig,
        f"{name} 運轉狀況",
        f"每分鐘資料，{df.index.min():%m/%d} ~ {df.index.max():%m/%d}",
        height=900,
    )


plot_operation("CH-1", df_ch1)
plot_operation("CH-2", df_ch2)

### 觀察

- 資料：09-17 13:23 ~ 10-13 11:24，每分鐘一筆
- CH-2：工作日開機；週末與 09-25、09-28、10-09（國定假日）停機；開機後約 1 小時為降溫暫態
- CH-1：僅 09-29 ~ 10-05 開機 5 天，均與 CH-2 同時運轉；出水溫度約 11°C

## 3. 冰水主機性能曲線

- 功率：`電錶即時功率(通訊)`；冷量：`冰水流量冷凍噸`
- 只用穩定運轉資料（kW > 100 且冷凍噸 > 50 RT）
- 負載率 = 冷凍噸 ÷ 額定（CH-1 375 RT、CH-2 400 RT）；效率 kW/RT 越低越省電

### 擬合式與指標

擬合式與 `onoff_control.ipynb` 相同：CH-1 為 kW/RT = a + b ÷ LR，CH-2 為 LR 的三次式；只用負載率 ≤ 100% 的資料擬合（超載時出水溫度失控）。指標彙整於本節最後的表格：

| 指標 | 意義 |
|---|---|
| R² | 擬合解釋 kW/RT 變化的比例，越接近 1 越好 |
| CV(RMSE) % | 均方根誤差 ÷ kW/RT 平均，可跨主機比較，越小越好 |

- 指標為 in-sample，且每分鐘資料前後高度相關，數字偏樂觀
- 擬合只在資料負載率範圍內有效；CH-1 負載率 55% 以上只有少數幾筆，高負載段不可靠

In [5]:
# kW/RT 擬合式文字（LR：負載率 %）
def equation(curve: dict) -> str:
    coef = curve["coef"]
    if curve["x"] == "1/LR":
        b, a = coef
        return f"kW/RT = {a:.3g} + {b:.3g}/LR"
    powers = {3: "·LR³", 2: "·LR²", 1: "·LR", 0: ""}
    degree = len(coef) - 1
    terms = []
    for i, value in enumerate(coef):
        if value < 0:
            sign = "−"
        else:
            sign = "+"
        terms.append(f"{sign} {abs(value):.3g}{powers[degree - i]}")
    text = " ".join(terms)
    # 第一項是正數時不寫「+」
    if text.startswith("+"):
        return "kW/RT = " + text[2:]
    return "kW/RT = " + "−" + text[2:]


# 性能曲線：每分鐘資料 + 每 5% 負載率平均 + 擬合曲線；回傳擬合式與指標
def draw_curve(name: str, df: pd.DataFrame) -> dict:
    curve = chiller.fit_curve(df, name)  # 只用負載率 ≤ 100% 擬合
    steady_data = chiller.steady(df, curve["rated"])
    fit_data = steady_data[steady_data["LR"] <= chiller.MAX_LR]
    residual = fit_data["KW_PER_RT"] - chiller.kw_per_rt(curve, fit_data["LR"])
    cv_rmse = np.sqrt((residual**2).mean()) / fit_data["KW_PER_RT"].mean() * 100

    lr_bins = pd.cut(steady_data["LR"], np.arange(0, 130, 5))
    bin_means = steady_data.groupby(lr_bins, observed=True)["KW_PER_RT"].agg(
        ["mean", "count"]
    )
    fit_lr = np.linspace(curve["lr_min"], curve["lr_max"], 100)
    fit_line = chiller.kw_per_rt(curve, fit_lr)

    fig = go.Figure()
    fig.add_trace(
        go.Scattergl(
            x=steady_data["LR"],
            y=steady_data["KW_PER_RT"],
            mode="markers",
            name="每分鐘資料",
            marker=dict(size=5, opacity=0.35, color=LIGHT_BLUE),
            text=steady_data.index.strftime("%m/%d %H:%M"),
            customdata=steady_data[["Q_COOL", "kW"]].values,
            hovertemplate=(
                "%{text}<br>Q_COOL: %{customdata[0]:.0f} RT<br>kW: %{customdata[1]:.0f}"
                "<br>負載率: %{x:.1f}%<br>kW/RT: %{y:.3f}<extra></extra>"
            ),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[load_bin.left + 2.5 for load_bin in bin_means.index],
            y=bin_means["mean"],
            mode="lines+markers",
            name="每 5% 負載率平均",
            line=dict(color=COLORS["KW_PER_RT"], width=2),
            marker=dict(size=8, line=dict(width=2, color="white")),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=fit_lr,
            y=fit_line,
            mode="lines",
            name=(
                f"擬合（R² {curve['r2']:.2f}、CV(RMSE) {cv_rmse:.1f}%）"
                f"<br>{equation(curve)}"
            ),
            line=dict(color=REF_COLOR, width=2, dash="dash"),
        )
    )
    # 圖例放圖內右上角，y 軸上方留白避免蓋到資料點
    y_top = steady_data["KW_PER_RT"].quantile(0.999)
    y_low = min(steady_data["KW_PER_RT"].quantile(0.001), fit_line.min())
    fig.update_yaxes(
        range=[y_low - 0.05 * (y_top - y_low), y_top + 0.45 * (y_top - y_low)]
    )
    fig.update_layout(
        xaxis_title=label("LR"),
        yaxis_title=label("KW_PER_RT"),
        legend=dict(
            orientation="v",
            x=0.99,
            xanchor="right",
            y=0.99,
            yanchor="top",
            bgcolor="rgba(255,255,255,0.85)",
        ),
    )
    show(
        fig,
        f"{name} 性能曲線",
        f"額定 {curve['rated']} RT，穩定運轉資料；擬合只用負載率 ≤ 100%",
    )

    print(
        f"{name}：穩定運轉 {len(steady_data)} 筆，"
        f"kW/RT 平均 {steady_data['KW_PER_RT'].mean():.3f}，"
        f"負載率 {steady_data['LR'].min():.0f}~{steady_data['LR'].max():.0f}%，"
        f"冷凍噸最高 {steady_data['Q_COOL'].max():.0f} RT"
    )
    print(bin_means.round(3).to_string())

    return {
        "主機": name,
        "擬合式": equation(curve),
        "擬合筆數": curve["n"],
        "擬合負載率範圍 %": f"{curve['lr_min']:.0f}~{curve['lr_max']:.0f}",
        "R²": curve["r2"],
        "CV(RMSE) %": cv_rmse,
    }


fit_table = pd.DataFrame(
    [
        draw_curve("CH-1", df_ch1),
        draw_curve("CH-2", df_ch2),
    ]
).set_index("主機")
fit_table.T

CH-1：穩定運轉 1693 筆，kW/RT 平均 1.260，負載率 24~83%，冷凍噸最高 313 RT
           mean  count
LR                    
(20, 25]  1.778      3
(25, 30]  1.596    203
(30, 35]  1.407    151
(35, 40]  1.255    862
(40, 45]  1.130    169
(45, 50]  1.071    199
(50, 55]  1.028     84
(55, 60]  0.922      6
(60, 65]  0.931      7
(65, 70]  0.848      2
(70, 75]  0.855      4
(80, 85]  0.812      3


CH-2：穩定運轉 7480 筆，kW/RT 平均 0.576，負載率 46~112%，冷凍噸最高 448 RT
             mean  count
LR                      
(45, 50]    0.793      2
(50, 55]    0.488    136
(55, 60]    0.465    508
(60, 65]    0.499    170
(65, 70]    0.515    510
(70, 75]    0.548    971
(75, 80]    0.579   1370
(80, 85]    0.605    849
(85, 90]    0.619    686
(90, 95]    0.620    822
(95, 100]   0.615   1124
(100, 105]  0.593    259
(105, 110]  0.568     71
(110, 115]  0.572      2


主機,CH-1,CH-2
擬合式,kW/RT = 0.305 + 35.9/LR,kW/RT = −6.33e-06·LR³ + 0.00139·LR² − 0.0958·L...
擬合筆數,1693,7148
擬合負載率範圍 %,24~83,46~100
R²,0.977868,0.692383
CV(RMSE) %,1.983376,5.517946


### 結論

- **CH-2**：穩定運轉 7,480 筆，kW/RT 平均 **0.576**（COP ≈ 6.1）；負載率 50~60% 最省電，85~100% 最差；45% 以下沒有資料
- **CH-1**：穩定運轉 1,693 筆，kW/RT 平均 **1.26**，約 CH-2 的 2.2 倍；負載率多在 25~55%，負載越低越耗電
- **超過額定**：CH-2 冷凍噸最高 448 RT，超過 400 RT 的資料佔 4%

## 4. 主機與氣象資料關聯

CH-1 + CH-2 合併：只取開機分鐘（功率 > 20 kW 且冰水流量 > 100 LPM；CH-1 停機時有待機耗電但沒有流量）；kW、冷凍噸、流量相加，出水／回水溫度依流量加權平均。

In [6]:
# 停機分鐘：功率、冷凍噸、流量設為 0，水溫設為空值；缺資料的分鐘維持空值
def running_only(chiller_minutes: pd.DataFrame) -> pd.DataFrame:
    chiller_minutes = chiller_minutes.copy()
    stopped = ~is_running(chiller_minutes) & chiller_minutes["kW"].notna()
    chiller_minutes.loc[stopped, ["kW", "Q_COOL", "FLOW"]] = 0
    chiller_minutes.loc[stopped, ["T_CHWS", "T_CHWR"]] = np.nan
    return chiller_minutes


# 合併兩台：kW、冷凍噸、流量相加，水溫依流量加權平均（兩台都停機時為空值）
def combine(ch1: pd.DataFrame, ch2: pd.DataFrame) -> pd.DataFrame:
    ch1 = running_only(ch1).reindex(ch2.index)
    ch2 = running_only(ch2)

    combined = pd.DataFrame(index=ch2.index)
    for column in ["kW", "Q_COOL", "FLOW"]:
        combined[column] = ch2[column] + ch1[column].fillna(0)
    for column in ["T_CHWS", "T_CHWR"]:
        ch2_heat = (ch2[column] * ch2["FLOW"]).fillna(0)
        ch1_heat = (ch1[column] * ch1["FLOW"]).fillna(0)
        combined[column] = (ch2_heat + ch1_heat) / combined["FLOW"]
    return combined


chiller_hourly = combine(df_ch1, df_ch2).resample("1h").mean()
chiller_hourly["DT"] = chiller_hourly["T_CHWR"] - chiller_hourly["T_CHWS"]

# 逐時氣象，加上濕球與焓值
start = int(chiller_hourly.index.min().timestamp())
# load_history 不含 end，多抓一小時
end = int(chiller_hourly.index.max().timestamp()) + 3_600
weather_hourly = weather.to_hourly(await weather.load_history(start, end))
weather_hourly["WET_BULB"] = weather.wet_bulb_temp(
    weather_hourly["temperature"],
    weather_hourly["humidity"],
)
weather_hourly["ENTHALPY"] = weather.enthalpy(
    weather_hourly["temperature"],
    weather_hourly["humidity"],
    weather_hourly["pressure"],
)

hourly_weather = pd.concat([chiller_hourly, weather_hourly], axis=1)
plot_df = hourly_weather[hourly_weather["FLOW"] > 2500]  # 正常運轉時段

# CH-1 該小時有開機
ch1_on = is_running(df_ch1).resample("1h").max().reindex(plot_df.index).fillna(False)
print("時間範圍：", hourly_weather.index.min(), "~", hourly_weather.index.max())
print(
    "正常運轉小時數：",
    len(plot_df),
    "（其中 CH-1 有開機：",
    int(ch1_on.sum()),
    "小時）",
)
print("有氣象資料的小時數：", plot_df["temperature"].notna().sum())

時間範圍： 2026-09-17 13:00:00+08:00 ~ 2026-10-13 11:00:00+08:00
正常運轉小時數： 134 （其中 CH-1 有開機： 33 小時）
有氣象資料的小時數： 134


### 時間序列總覽（每小時，含停機時段）

In [7]:
# 各機逐時平均（同 combine()，停機分鐘算 0）
def hourly_by_chiller(column: str) -> dict:
    chiller_minutes = [("CH-1", df_ch1), ("CH-2", df_ch2)]
    return {
        name: running_only(minutes)[column]
        .resample("1h")
        .mean()
        .reindex(hourly_weather.index)
        for name, minutes in chiller_minutes
    }


# 同色系：CH-1 淺色、CH-2 深色、合計標準色虛線
SHADES = {
    "Q_COOL": {"CH-1": "#86b6ef", "CH-2": "#104281", "合計": "#2a78d6"},
    "kW": {"CH-1": "#f5a582", "CH-2": "#a33f17", "合計": "#eb6834"},
}
DASHES = {"CH-1": None, "CH-2": None, "合計": "dash"}

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.09,
    subplot_titles=("外氣溫度：乾球 / 濕球 (°C)", label("Q_COOL"), label("kW")),
)

# 第一排：外氣乾球、濕球
for column, dash in [("temperature", None), ("WET_BULB", "dot")]:
    fig.add_trace(
        go.Scatter(
            x=hourly_weather.index,
            y=hourly_weather[column],
            name=NAMES[column],
            legend="legend",
            line=dict(color=COLORS[column], width=2, dash=dash),
        ),
        row=1,
        col=1,
    )

# 第二、三排：冷凍噸、耗電，各畫 CH-1、CH-2、合計
for row, column, legend in [(2, "Q_COOL", "legend2"), (3, "kW", "legend3")]:
    lines = hourly_by_chiller(column)
    lines["合計"] = lines["CH-1"] + lines["CH-2"]
    for line_name, line_values in lines.items():
        fig.add_trace(
            go.Scatter(
                x=line_values.index,
                y=line_values,
                name=line_name,
                legend=legend,
                line=dict(
                    color=SHADES[column][line_name],
                    width=2,
                    dash=DASHES[line_name],
                ),
            ),
            row=row,
            col=1,
        )

# 每排的圖例放在該子圖右上角
for i, legend in enumerate(["legend", "legend2", "legend3"], start=1):
    if i == 1:
        axis_name = "yaxis"
    else:
        axis_name = f"yaxis{i}"
    top = fig.layout[axis_name].domain[1]
    fig.update_layout(
        {legend: dict(orientation="h", x=1, xanchor="right", y=top, yanchor="bottom")}
    )

fig.update_layout(hovermode="x unified")
date_axis(fig)
show(fig, "主機（CH-1 / CH-2）與外氣時間序列", "每小時平均（含停機）", height=900)

### 散佈圖（正常運轉時段）

In [8]:
# R²：越接近 1，兩者的直線關係越強
def r2(data: pd.DataFrame, x_col: str, y_col: str) -> float:
    pair = data[[x_col, y_col]].dropna()
    return pair[x_col].corr(pair[y_col]) ** 2


# 依 CH-1 有沒有同時運轉分成兩組
def split_by_ch1(data: pd.DataFrame) -> dict:
    ch1_running = ch1_on.reindex(data.index).fillna(False).astype(bool)
    return {
        "只有 CH-2 運轉": data[~ch1_running],
        "CH-1 + CH-2 同時運轉": data[ch1_running],
    }


# 依運轉狀態畫兩組散佈點，顏色用 y 軸那個量的深淺
def add_grouped_scatter(
    fig: go.Figure,
    data: pd.DataFrame,
    x_col: str,
    y_col: str,
    row: int | None = None,
    col: int | None = None,
    showlegend: bool = True,
) -> None:
    pair = data[[x_col, y_col]].dropna()
    for group_name, group in split_by_ch1(pair).items():
        fig.add_trace(
            go.Scattergl(
                x=group[x_col],
                y=group[y_col],
                mode="markers",
                name=group_name,
                legendgroup=group_name,
                showlegend=showlegend,
                marker=dict(
                    size=8,
                    color=GROUP_SHADES[y_col][group_name],
                    opacity=0.8,
                    line=dict(width=1, color="white"),
                ),
                text=group.index.strftime("%m/%d %H:%M"),
                hovertemplate=(
                    f"%{{text}}　{group_name}"
                    f"<br>{NAMES[x_col]}: %{{x:.1f}} {UNITS[x_col]}"
                    f"<br>{NAMES[y_col]}: %{{y:.1f}} {UNITS[y_col]}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )


# 單張散佈圖，副標題標 R²
def draw_scatter(data: pd.DataFrame, x_col: str, y_col: str) -> None:
    fig = go.Figure()
    add_grouped_scatter(fig, data, x_col, y_col)
    fig.update_layout(xaxis_title=label(x_col), yaxis_title=label(y_col))
    show(
        fig,
        f"{NAMES[y_col].split()[0]} vs {NAMES[x_col].split()[0]}",
        f"R² = {r2(data, x_col, y_col):.2f}，正常運轉時段",
        legend="bottom",
    )


# 左右對照散佈圖（正常運轉時段），子圖標題標 R²；兩張子圖點相同，圖例只在第 legend_at 張顯示
def draw_pair(
    y_col: str,
    x_cols: list,
    title: str,
    legend_at: int = 0,
    **subplot_kw,
) -> None:
    subplot_titles = []
    for x_col in x_cols:
        subplot_titles.append(
            f"vs {NAMES[x_col].split()[0]}（R² = {r2(plot_df, x_col, y_col):.2f}）"
        )
    fig = make_subplots(rows=1, cols=2, subplot_titles=subplot_titles, **subplot_kw)
    for i, x_col in enumerate(x_cols):
        add_grouped_scatter(
            fig,
            plot_df,
            x_col,
            y_col,
            row=1,
            col=i + 1,
            showlegend=(i == legend_at),
        )
        fig.update_xaxes(title_text=label(x_col), row=1, col=i + 1)
    fig.update_yaxes(title_text=label(y_col), row=1, col=1)
    show(fig, title, "正常運轉時段", legend="bottom")

In [9]:
# 外氣越熱，冷負載越高？乾球、濕球左右對照，y 軸共用
draw_pair(
    "Q_COOL",
    ["temperature", "WET_BULB"],
    "冷凍噸 vs 外氣乾球 / 濕球溫度",
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

print(
    "冷凍噸的 R²：乾球",
    round(r2(plot_df, "temperature", "Q_COOL"), 2),
    "／濕球",
    round(r2(plot_df, "WET_BULB", "Q_COOL"), 2),
)

冷凍噸的 R²：乾球 0.34 ／濕球 0.71


In [10]:
# 出水溫度有沒有隨外氣或負載調整？（出水溫度 reset 的空間）
draw_pair(
    "T_CHWS",
    ["temperature", "Q_COOL"],
    "出水溫度 vs 外氣乾球溫度 / 負載",
    legend_at=1,
    horizontal_spacing=0.08,
)

supply_temp = plot_df["T_CHWS"]
print(
    "T_CHWS 的 R²：乾球",
    round(r2(plot_df, "temperature", "T_CHWS"), 2),
    "／冷凍噸",
    round(r2(plot_df, "Q_COOL", "T_CHWS"), 2),
)
print(
    "T_CHWS 最低／中位／最高：",
    round(supply_temp.min(), 2),
    round(supply_temp.median(), 2),
    round(supply_temp.max(), 2),
)
print()

# 兩種運轉狀態分開看
groups = split_by_ch1(plot_df)
ch2_only = groups["只有 CH-2 運轉"]
both = groups["CH-1 + CH-2 同時運轉"]
print(
    f"只有 CH-2 運轉（{len(ch2_only)} 小時）"
    f"：T_CHWS 中位 {ch2_only['T_CHWS'].median():.2f}°C，"
    f"範圍 {ch2_only['T_CHWS'].min():.2f}~{ch2_only['T_CHWS'].max():.2f}°C，"
    f"跟冷凍噸的 R² = {r2(ch2_only, 'Q_COOL', 'T_CHWS'):.2f}，"
    f"冷凍噸平均 {ch2_only['Q_COOL'].mean():.0f} RT"
)
print(
    f"CH-1 + CH-2 同時運轉（{len(both)} 小時）"
    f"：T_CHWS 中位 {both['T_CHWS'].median():.2f}°C，"
    f"冷凍噸平均 {both['Q_COOL'].mean():.0f} RT"
)
ch1_running_supply = df_ch1.loc[is_running(df_ch1), "T_CHWS"].median()
print(f"CH-1 本身開機時的出水溫度中位：{ch1_running_supply:.2f}°C")
print()

# 只有 CH-2 運轉：出水 ≤ 9°C vs 偏高；偏高時段多半是接近滿載（≥ 360 RT）或早上 8 點剛開機
low_supply = ch2_only[ch2_only["T_CHWS"] <= 9.0]
high_supply = ch2_only[ch2_only["T_CHWS"] > 9.0]
explained = (high_supply["Q_COOL"] >= 360) | (high_supply.index.hour == 8)
print(
    f"只有 CH-2、出水 ≤ 9°C：{len(low_supply)} 小時，"
    f"出水 {low_supply['T_CHWS'].min():.2f}~{low_supply['T_CHWS'].max():.2f}°C，"
    f"冷凍噸 {low_supply['Q_COOL'].min():.0f}~{low_supply['Q_COOL'].max():.0f} RT，"
    f"跟冷凍噸的 R² = {r2(low_supply, 'Q_COOL', 'T_CHWS'):.2f}"
)
print(
    f"只有 CH-2、出水 > 9°C：{len(high_supply)} 小時，"
    f"出水 {high_supply['T_CHWS'].min():.2f}~{high_supply['T_CHWS'].max():.2f}°C，"
    f"其中冷凍噸 ≥ 360 RT 或早上 8 點開機的有 {int(explained.sum())} 小時"
)

T_CHWS 的 R²：乾球 0.03 ／冷凍噸 0.53
T_CHWS 最低／中位／最高： 8.47 8.81 12.1

只有 CH-2 運轉（101 小時）：T_CHWS 中位 8.68°C，範圍 8.47~12.10°C，跟冷凍噸的 R² = 0.26，冷凍噸平均 306 RT
CH-1 + CH-2 同時運轉（33 小時）：T_CHWS 中位 9.89°C，冷凍噸平均 444 RT
CH-1 本身開機時的出水溫度中位：11.00°C

只有 CH-2、出水 ≤ 9°C：73 小時，出水 8.47~8.97°C，冷凍噸 182~377 RT，跟冷凍噸的 R² = 0.01
只有 CH-2、出水 > 9°C：28 小時，出水 9.02~12.10°C，其中冷凍噸 ≥ 360 RT 或早上 8 點開機的有 24 小時


### 發現（正常運轉 134 小時，其中 CH-1 開機 33 小時）

- **外氣 vs 負載**：濕球 R² = 0.71、乾球 R² = 0.34，濕球較能解釋負載
- **出水溫度**：與外氣無關（R² = 0.03）；與冷凍噸 R² = 0.53，是兩種情況疊加（右圖）：
  - 只有 CH-2 運轉：多在 8.5~9.0°C，不隨負載變化；出水高於 9°C 的 28 小時中，24 小時為 360 RT 以上（約額定 90%）或剛開機
  - CH-1 + CH-2：CH-1 出水約 11°C，把合併出水拉到約 9.9°C
- → 出水溫度設定固定，未隨外氣或負載調整，有 reset 節能空間

## 5. 外氣焓值 vs 冰水溫差 vs 耗電

焓值綜合乾球溫度、濕度與氣壓，代表外氣帶進的總熱量。資料同第 4 節。

In [11]:
enthalpy_data = plot_df[["ENTHALPY", "DT", "kW"]].dropna()
fig = go.Figure()
for group_name, group in split_by_ch1(enthalpy_data).items():
    fig.add_trace(
        go.Scatter3d(
            x=group["ENTHALPY"],
            y=group["DT"],
            z=group["kW"],
            mode="markers",
            name=group_name,
            marker=dict(size=4, color=GROUP_SHADES["kW"][group_name], opacity=0.8),
            text=group.index.strftime("%m/%d %H:%M"),
            hovertemplate=(
                f"%{{text}}　{group_name}<br>焓值: %{{x:.1f}} kJ/kg"
                "<br>冰水溫差: %{y:.1f} °C<br>kW: %{z:.1f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    scene=dict(
        xaxis_title=label("ENTHALPY"),
        yaxis_title=label("DT"),
        zaxis_title=label("kW"),
        # 放大並往上移
        camera=dict(eye=dict(x=1.35, y=1.35, z=0.85), center=dict(x=0, y=0, z=-0.2)),
    ),
    margin=dict(l=0, r=0, t=80, b=0),  # 3D 圖不需要座標軸邊界，減少空白
    # 右上角
    legend=dict(
        orientation="v",
        x=0.99,
        xanchor="right",
        y=0.99,
        yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
    ),
)
show(fig, "外氣焓值 vs 冰水溫差 vs 耗電", "正常運轉時段", height=720)

In [12]:
draw_scatter(plot_df, "ENTHALPY", "kW")
print("耗電 vs 焓值 R²：", round(r2(plot_df, "ENTHALPY", "kW"), 2))
print(plot_df[["ENTHALPY", "kW", "DT"]].describe().round(2).to_string())

耗電 vs 焓值 R²： 0.69
       ENTHALPY      kW      DT
count    134.00  134.00  134.00
mean      71.48  215.68    3.94
std        7.89   86.74    0.62
min       51.91   84.47    2.70
25%       64.81  145.35    3.49
50%       72.16  212.96    3.96
75%       78.58  250.32    4.36
max       84.39  405.30    5.44


### 發現

- **焓值 vs 耗電：R² = 0.69**；焓值 52~84 kJ/kg，耗電 84~405 kW
- 淺色點（CH-1 + CH-2）整群在深色點上方：同焓值下 CH-1 一起開時耗電明顯較高，對應 CH-1 效率差（kW/RT 1.26）
- **冰水溫差**：2.7~5.4°C（中位 4.0°C），變化不大

## 6. 總結

- **運轉**：CH-2 工作日固定運轉；CH-1 只在 09-29 ~ 10-05 與 CH-2 同時開 5 天
- **性能**：CH-2 kW/RT 0.576，負載率 50~60% 最省電；CH-1 kW/RT 1.26，效率明顯較差
- **氣象**：負載與濕球 R² = 0.71，耗電與焓值 R² = 0.69
- **出水溫度**：設定固定（只有 CH-2 運轉時多在 8.5~9.0°C），未隨外氣或負載調整